# Egyptian Hieroglyphs Recognition

This notebook trains an ImageNet transfer-learning classifier from cropped object-detection annotations. It is intentionally data-dependent: no dataset or metric result is bundled with this repository.

**Run cells from top to bottom.** Set `HIEROGLYPHS_DATA_DIR` to a directory containing `train/`, `valid/`, and `test/`; each split must contain `_annotations.csv` and the referenced image files. The notebook fails early with an actionable message when this contract is not met.

In [ ]:
import os
import random
from pathlib import Path

os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from tqdm.auto import tqdm

SEED = int(os.environ.get("HIEROGLYPHS_SEED", "42"))
IMAGE_SIZE = int(os.environ.get("HIEROGLYPHS_IMAGE_SIZE", "224"))
BATCH_SIZE = int(os.environ.get("HIEROGLYPHS_BATCH_SIZE", "16"))
EPOCHS = int(os.environ.get("HIEROGLYPHS_EPOCHS", "20"))
MODEL_NAME = os.environ.get("HIEROGLYPHS_MODEL", "ResNet50")
USE_IMAGENET_WEIGHTS = os.environ.get("HIEROGLYPHS_USE_IMAGENET_WEIGHTS", "1") == "1"

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError):
    # Older TensorFlow versions or unsupported devices may not expose this API.
    pass

print(f"TensorFlow {tf.__version__}; seed={SEED}; model={MODEL_NAME}")


## Data contract and validation

The expected layout is:

```text
$HIEROGLYPHS_DATA_DIR/
├── train/_annotations.csv
├── valid/_annotations.csv
└── test/_annotations.csv
```

Each CSV must contain `filename`, `class`, `xmin`, `ymin`, `xmax`, and `ymax`. Bounding boxes are validated before any image is loaded. Class IDs are fit **only on the training split**; validation and test labels must be known training classes. Exact filename overlap across splits is rejected as a leakage safeguard.

In [ ]:
DATA_ROOT = Path(os.environ.get("HIEROGLYPHS_DATA_DIR", "data/egyptian-hieroglyphs")).expanduser().resolve()
SPLITS = ("train", "valid", "test")
ANNOTATION_COLUMNS = ("filename", "class", "xmin", "ymin", "xmax", "ymax")

def read_and_validate_annotations(split: str) -> pd.DataFrame:
    """Read one split and validate the annotation schema and bounding boxes."""
    split_dir = DATA_ROOT / split
    annotation_path = split_dir / "_annotations.csv"
    if not annotation_path.is_file():
        raise FileNotFoundError(
            f"Missing {annotation_path}. Set HIEROGLYPHS_DATA_DIR to the dataset "
            "root containing train/, valid/, test/ and each _annotations.csv."
        )

    frame = pd.read_csv(annotation_path).reset_index(drop=True)
    missing = sorted(set(ANNOTATION_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(f"{annotation_path} is missing required columns: {missing}")
    if frame.empty:
        raise ValueError(f"{annotation_path} contains no annotations")

    if frame["filename"].isna().any() or frame["class"].isna().any():
        raise ValueError(f"{annotation_path} contains missing filename or class values")
    frame["filename"] = frame["filename"].astype(str)
    frame["class"] = frame["class"].astype(str)
    if frame["filename"].str.strip().eq("").any() or frame["class"].str.strip().eq("").any():
        raise ValueError(f"{annotation_path} contains blank filename or class values")

    for column in ("xmin", "ymin", "xmax", "ymax"):
        numeric = pd.to_numeric(frame[column], errors="coerce")
        if numeric.isna().any() or not np.allclose(numeric, numeric.round()):
            raise ValueError(f"{annotation_path} has non-integer or missing {column} values")
        frame[column] = numeric.astype(np.int64)

    valid_boxes = (
        (frame["xmin"] >= 0)
        & (frame["ymin"] >= 0)
        & (frame["xmax"] > frame["xmin"])
        & (frame["ymax"] > frame["ymin"])
    )
    if not valid_boxes.all():
        bad_rows = frame.index[~valid_boxes].tolist()[:5]
        raise ValueError(f"{annotation_path} has invalid bounding boxes at rows {bad_rows}")
    return frame

annotations = {split: read_and_validate_annotations(split) for split in SPLITS}
seen_filenames = set()
for split, frame in annotations.items():
    duplicate = seen_filenames.intersection(frame["filename"])
    if duplicate:
        raise ValueError(
            f"Potential split leakage: filenames occur in more than one split, e.g. {sorted(duplicate)[:3]}"
        )
    seen_filenames.update(frame["filename"])

class_names = sorted(annotations["train"]["class"].unique().tolist())
if not class_names:
    raise ValueError("The training annotations contain no classes")
class_to_id = {name: index for index, name in enumerate(class_names)}
unknown_classes = {
    split: sorted(set(frame["class"]) - set(class_names))
    for split, frame in annotations.items()
    if split != "train"
}
unknown_classes = {split: values for split, values in unknown_classes.items() if values}
if unknown_classes:
    raise ValueError(f"Validation/test contain classes absent from train: {unknown_classes}")

print({split: len(frame) for split, frame in annotations.items()})
print(f"Classes learned from train only: {len(class_names)}")


In [ ]:
def load_split(split: str, frame: pd.DataFrame):
    """Load and crop images for one validated split."""
    images = []
    labels = []
    split_dir = DATA_ROOT / split
    columns = ["filename", "class", "xmin", "ymin", "xmax", "ymax"]
    for filename, label, xmin, ymin, xmax, ymax in tqdm(
        frame[columns].itertuples(index=False, name=None),
        total=len(frame),
        desc=f"Loading {split}",
    ):
        image_path = (split_dir / filename).resolve()
        try:
            image_path.relative_to(split_dir.resolve())
        except ValueError as exc:
            raise ValueError(f"Annotation path escapes its split directory: {filename}") from exc
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Could not read image referenced by annotations: {image_path}")
        height, width = image.shape[:2]
        if xmax > width or ymax > height:
            raise ValueError(
                f"Bounding box for {image_path} exceeds image bounds ({width}x{height}): "
                f"({xmin}, {ymin}, {xmax}, {ymax})"
            )
        crop = image[ymin:ymax, xmin:xmax]
        if crop.size == 0:
            raise ValueError(f"Empty crop for {image_path}")
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        crop = cv2.resize(crop, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
        images.append(crop)
        labels.append(class_to_id[str(label)])
    return np.stack(images).astype(np.uint8), np.asarray(labels, dtype=np.int32)
x_train, y_train = load_split("train", annotations["train"])
x_valid, y_valid = load_split("valid", annotations["valid"])
x_test, y_test = load_split("test", annotations["test"])
print(x_train.shape, x_valid.shape, x_test.shape)


In [ ]:
def show_samples(images, labels, title, seed=SEED, count=16):
    """Display a deterministic sample without changing the training data."""
    rng = np.random.default_rng(seed)
    selected = rng.choice(len(images), size=min(count, len(images)), replace=False)
    columns = 4
    rows = int(np.ceil(len(selected) / columns))
    plt.figure(figsize=(12, 3 * rows))
    for position, index in enumerate(selected, start=1):
        plt.subplot(rows, columns, position)
        plt.imshow(images[index])
        plt.title(class_names[labels[index]])
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_samples(x_train, y_train, "Training samples")


## Transfer-learning model

The classifier head uses the number of classes discovered in the training annotations rather than a hard-coded value. ImageNet weights are optional because downloading them may be unavailable in a restricted environment; set `HIEROGLYPHS_USE_IMAGENET_WEIGHTS=0` for an explicitly uninitialized backbone (not comparable to ImageNet transfer learning).

In [ ]:
MODEL_CONFIGS = {
    "ResNet50": (tf.keras.applications.ResNet50, tf.keras.applications.resnet50.preprocess_input),
    "ResNet152": (tf.keras.applications.ResNet152, tf.keras.applications.resnet.preprocess_input),
    "MobileNetV2": (tf.keras.applications.MobileNetV2, tf.keras.applications.mobilenet_v2.preprocess_input),
    "EfficientNetB0": (tf.keras.applications.EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input),
}
if MODEL_NAME not in MODEL_CONFIGS:
    raise ValueError(f"Unsupported HIEROGLYPHS_MODEL={MODEL_NAME!r}; choose from {sorted(MODEL_CONFIGS)}")

backbone_builder, preprocess_input = MODEL_CONFIGS[MODEL_NAME]
weights = "imagenet" if USE_IMAGENET_WEIGHTS else None
inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="image")
base_model = backbone_builder(include_top=False, weights=weights, input_tensor=inputs)
base_model.trainable = False
features = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
features = tf.keras.layers.Dense(256, activation="relu")(features)
features = tf.keras.layers.BatchNormalization()(features)
features = tf.keras.layers.Dropout(0.3)(features)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax", name="class_probabilities")(features)
model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

x_train_model = preprocess_input(x_train.astype(np.float32))
x_valid_model = preprocess_input(x_valid.astype(np.float32))
x_test_model = preprocess_input(x_test.astype(np.float32))


In [ ]:
ARTIFACT_DIR = Path(os.environ.get("HIEROGLYPHS_ARTIFACT_DIR", "artifacts")).expanduser()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
best_model_path = ARTIFACT_DIR / "best_model.keras"
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(str(best_model_path), monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6),
]
history = model.fit(
    x_train_model,
    y_train,
    epochs=EPOCHS,
    validation_data=(x_valid_model, y_valid),
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    shuffle=True,
    verbose=1,
)


In [ ]:
history_frame = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_frame["accuracy"], label="train")
axes[0].plot(history_frame["val_accuracy"], label="valid")
axes[0].set_title("Accuracy")
axes[0].legend()
axes[1].plot(history_frame["loss"], label="train")
axes[1].plot(history_frame["val_loss"], label="valid")
axes[1].set_title("Loss")
axes[1].legend()
plt.tight_layout()
plt.show()


## Final test evaluation

The validation split is used during training and checkpoint selection. The test split is evaluated only here and is not used for fitting, model selection, or preprocessing. The metrics below are produced only when the dataset and dependencies are available; this repository does not claim precomputed results.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test_model, y_test, verbose=0)
test_probabilities = model.predict(x_test_model, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)

metrics = {
    "loss": float(test_loss),
    "accuracy": accuracy_score(y_test, test_predictions),
    "precision_weighted": precision_score(y_test, test_predictions, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, test_predictions, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, test_predictions, average="weighted", zero_division=0),
}
print(pd.Series(metrics))
print(
    classification_report(
        y_test,
        test_predictions,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0,
    )
)
